# Tarea 2

In [1]:
# Este código fue ejecutado localmente en un entorno virtual con anaconda, es necesario tener instalado:
# PARA CUDA: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# PARA DEPENDENCIAS: pip install gdown opencv-python torch torchvision pillow numpy pandas scikit-learn scipy

import gdown
import zipfile
import cv2
import torch
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from sklearn.metrics import f1_score
import os
import sys

In [2]:
# Se verifica al inicio si esta disponible CUDA porque es vital para procesar todos los videos más adelante
print(f"Python executable: {sys.executable}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"Versión de CUDA: {torch.version.cuda}")

Python executable: c:\Users\carit\anaconda3\envs\entornoTAV\python.exe
CUDA disponible: True
Versión de CUDA: 12.1


### Descarga de colección de datos
Links de Drive usados:
- BDSumm: https://drive.google.com/file/d/1K-aIkr27WZ829iMHFoC3tgv16wN87XBM/view?usp=sharing
- OpenVideo: https://drive.google.com/file/d/1NBOoeW77TQiOKeOWDVDwUl6mIlC5ievd/view?usp=sharing


In [3]:
# Descargar colección de datos BDSumm
url1 = "https://drive.google.com/uc?id=1K-aIkr27WZ829iMHFoC3tgv16wN87XBM"
output1 = "BDSumm.zip"
gdown.download(url1, output=output1)

# Descomprimir archivo
datasetDir = "DatosTarea2"
with zipfile.ZipFile(output1, 'r') as zip_ref:
    zip_ref.extractall(datasetDir)

print(f"\nEl archivo {output1} ha sido descargado y descomprimido en la carpeta {datasetDir}.")

Downloading...
From (original): https://drive.google.com/uc?id=1K-aIkr27WZ829iMHFoC3tgv16wN87XBM
From (redirected): https://drive.google.com/uc?id=1K-aIkr27WZ829iMHFoC3tgv16wN87XBM&confirm=t&uuid=da111c45-deda-4022-9f5d-ef158fe493f4
To: c:\Users\carit\Dropbox\Carolina Universidad\2024-2\Análisis de video (Tópico I)\Tarea 2\BDSumm.zip
100%|██████████| 177M/177M [00:04<00:00, 37.7MB/s] 



El archivo BDSumm.zip ha sido descargado y descomprimido en la carpeta DatosTarea2.


In [4]:
# Descargar colección de datos OpenVideo
url2 = "https://drive.google.com/uc?id=1NBOoeW77TQiOKeOWDVDwUl6mIlC5ievd"
output2 = "OpenVideo.zip"
gdown.download(url2, output=output2)

# Descomprimir archivo
with zipfile.ZipFile(output2, 'r') as zip_ref:
    zip_ref.extractall(datasetDir)

print(f"\nEl archivo {output2} ha sido descargado y descomprimido en la carpeta {datasetDir}.")

Downloading...
From (original): https://drive.google.com/uc?id=1NBOoeW77TQiOKeOWDVDwUl6mIlC5ievd
From (redirected): https://drive.google.com/uc?id=1NBOoeW77TQiOKeOWDVDwUl6mIlC5ievd&confirm=t&uuid=916d0f48-960e-4db3-b3c5-5b4414e1a111
To: c:\Users\carit\Dropbox\Carolina Universidad\2024-2\Análisis de video (Tópico I)\Tarea 2\OpenVideo.zip
100%|██████████| 368M/368M [00:09<00:00, 40.7MB/s] 



El archivo OpenVideo.zip ha sido descargado y descomprimido en la carpeta DatosTarea2.


## Ejercicio 1 
Computar los sumarios estáticos de los videos que forman parte de las colecciones BDSumm y OpenVideo usando el método K-means. Debe realizar un estudio experimental para determinar el mejor valor de K (probar con K = 3, K = 5 y K=7) para obtener los sumarios en cada colección de datos.

### Leer videos y extraer frames

In [5]:
# OBJETIVO FUNCIÓN: Leer videos y extraer sus frames
def read_videos(folder_path):
    video_files = [] # Lista para almacenar los videos
    
    # Se recorre cada video y se añade su ruta a la lista
    for f in os.listdir(folder_path):
        if f.endswith(".mp4"):
            file_path = os.path.join(folder_path, f)
            video_files.append(file_path)

    all_frames = [] # Lista para almacenar los frames (lista de listas, cada sublista tiene los frames de un video)

    # Se procesa cada video
    for video_path in video_files:
        # Lectura de video
        vid_capture = cv2.VideoCapture(video_path)

        frames = [] # Lista de frames del video actual

        # Lectura de frames
        while vid_capture.isOpened():
            ret, frame = vid_capture.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
            else:
                break
        
        # Se cierra el video y se añade sus frames a la lista de todos los frames
        vid_capture.release()
        all_frames.append(frames)
        print(f"Frames extraídos de {os.path.basename(video_path)}: {len(frames)}")

    return all_frames

In [6]:
# Se definen las rutas de los videos para cada colección de datos
bdsumm_path = os.path.join(datasetDir, "BDSumm")
openvideo_path = os.path.join(datasetDir, "OpenVideoDB")

# Se leen los videos y se extraen los frames (este proceso se demora como 7 min en mi pc)
bdsumm_frames = read_videos(bdsumm_path)
openvideo_frames = read_videos(openvideo_path)

print(f"\nTotal de videos en BDSumm: {len(bdsumm_frames)}")
print(f"Total de videos en OpenVideo: {len(openvideo_frames)}")

Frames extraídos de v11.mp4: 2342
Frames extraídos de v12.mp4: 1505
Frames extraídos de v13.mp4: 3918
Frames extraídos de v14.mp4: 3452
Frames extraídos de v15.mp4: 2558
Frames extraídos de v16.mp4: 2650
Frames extraídos de v17.mp4: 5090
Frames extraídos de v18.mp4: 3450
Frames extraídos de v19.mp4: 2645
Frames extraídos de v20.mp4: 3100
Frames extraídos de v21.mp4: 1339
Frames extraídos de v22.mp4: 3180
Frames extraídos de v23.mp4: 5043
Frames extraídos de v24.mp4: 3331
Frames extraídos de v25.mp4: 1823
Frames extraídos de v26.mp4: 1230
Frames extraídos de v27.mp4: 1151
Frames extraídos de v28.mp4: 935
Frames extraídos de v29.mp4: 1580
Frames extraídos de v30.mp4: 2628
Frames extraídos de v21.mp4: 3284
Frames extraídos de v22.mp4: 2118
Frames extraídos de v23.mp4: 1759
Frames extraídos de v24.mp4: 1815
Frames extraídos de v25.mp4: 1797
Frames extraídos de v26.mp4: 6260
Frames extraídos de v27.mp4: 3185
Frames extraídos de v28.mp4: 3561
Frames extraídos de v29.mp4: 1944
Frames extraído

### Implementación ResNet18 y extraer descriptores

In [7]:
# Se carga modelo ResNet18 preentrenado
model = models.resnet18(pretrained=True)

# Se habilita el modo de evaluación
model.eval()

# Se manda el modelo al GPU si es posible, de lo contrario se usa CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Nombre de la GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

# Mover el modelo al dispositivo seleccionado
model.to(device)

print(f"Dispositivo seleccionado: {device}")

c:\Users\carit\anaconda3\envs\entornoTAV\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\carit\anaconda3\envs\entornoTAV\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Nombre de la GPU: NVIDIA GeForce GTX 1650
Dispositivo seleccionado: cuda


In [8]:
# Se prepara la transformación que se debe aplicar a cada frame
transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [9]:
# OBJETIVO FUNCIÓN: Extraer descriptores de todos los videos de una colección (utilizando procesamiento en lotes)
def extract_descriptors_batch(frames, batch_size):
    all_descriptors = []

    for i in range(0, len(frames), batch_size):
        batch_frames = frames[i:i + batch_size]

        # Se convierte las imagenes a tensores y se apilan en un batch
        batch_tensors = torch.stack(
            [transform(Image.fromarray(frame)).to(device) for frame in batch_frames]
        )

        with torch.no_grad():
            # Se procesa el batch en ResNet18
            outputs = model(batch_tensors)
            # Se pasa a CPU para manejar con numpy
            outputs = outputs.cpu().numpy()

        all_descriptors.append(outputs)
        print(f"Procesado batch {i // batch_size + 1} de {len(frames) // batch_size + 1}")

    # Se combina todos los descriptores en una sola matriz
    descriptors = np.vstack(all_descriptors)
    return descriptors

In [10]:
# Notar que la lista de frames contiene sublistas
# para optimizar computacionalmente y por simplicidad, en la extracción de descriptores se van a aplanar todos los frames a una lista continua

# Listas planas de BDSumm y OpenVideo
bdsumm_frames_flat = []
openvideo_frames_flat = []

# Se aplana la lista de BDSumm
for video in bdsumm_frames:
    for frame in video:
        bdsumm_frames_flat.append(frame)

# Se aplana la lista de OpenVideo
for video in openvideo_frames:
    for frame in video:
        openvideo_frames_flat.append(frame)

print(f"Frames en BDSumm: {len(bdsumm_frames_flat)}")
print(f"Frames en OpenVideo: {len(openvideo_frames_flat)}")

Frames en BDSumm: 52950
Frames en OpenVideo: 149210


In [ ]:
# Se extraen los descriptores para BDSumm y OpenVideo (este proceso se demora bastante como 25 min en mi pc usando cudas)
bdsumm_descriptors = extract_descriptors_batch(bdsumm_frames_flat, batch_size=32)
openvideo_descriptors = extract_descriptors_batch(openvideo_frames_flat, batch_size=32)

print(f"Descriptores de BDSumm extraídos: {len(bdsumm_descriptors)} frames procesados.")
print(f"Descriptores de OpenVideo extraídos: {len(openvideo_descriptors)} frames procesados.")

Procesado batch 1 de 1655
Procesado batch 2 de 1655
Procesado batch 3 de 1655
Procesado batch 4 de 1655
Procesado batch 5 de 1655
Procesado batch 6 de 1655
Procesado batch 7 de 1655
Procesado batch 8 de 1655
Procesado batch 9 de 1655
Procesado batch 10 de 1655
Procesado batch 11 de 1655
Procesado batch 12 de 1655
Procesado batch 13 de 1655
Procesado batch 14 de 1655
Procesado batch 15 de 1655
Procesado batch 16 de 1655
Procesado batch 17 de 1655
Procesado batch 18 de 1655
Procesado batch 19 de 1655
Procesado batch 20 de 1655
Procesado batch 21 de 1655
Procesado batch 22 de 1655
Procesado batch 23 de 1655
Procesado batch 24 de 1655
Procesado batch 25 de 1655
Procesado batch 26 de 1655
Procesado batch 27 de 1655
Procesado batch 28 de 1655
Procesado batch 29 de 1655
Procesado batch 30 de 1655
Procesado batch 31 de 1655
Procesado batch 32 de 1655
Procesado batch 33 de 1655
Procesado batch 34 de 1655
Procesado batch 35 de 1655
Procesado batch 36 de 1655
Procesado batch 37 de 1655
Procesado 

### Procesar anotaciones de las colecciones de datos

In [12]:
# Se procesan las anotaciones (sumarios humanos) de BDSumm:

# Se define la ruta de las anotaciones
user_bdsumm_path = os.path.join(datasetDir, "UserBDSumm")

# Se utiliza un diccionario para almacenar las anotaciones
bdsumm_annotations = {}

# Lectura de anotaciones
for folder in os.listdir(user_bdsumm_path):
    folder_path = os.path.join(user_bdsumm_path, folder)

    # Se mete en carpeta v11 a la v30
    if os.path.isdir(folder_path):
        txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]
        bdsumm_annotations[folder] = {}

        print(f"Archivos en {folder_path}: {txt_files}")

        # Lectura de los archivos .txt US-1, US-2 y US-3
        for txt_file in txt_files:
            txt_path = os.path.join(folder_path, txt_file)
            
            # Se almacenan las anotaciones de frames de un usuario en especifico
            with open(txt_path, "r") as file:
                frames = [int(line.strip()) for line in file.readlines()]
                bdsumm_annotations[folder][txt_file] = frames

print(f"\nAnotaciones de BDSumm: {len(bdsumm_annotations)} videos procesados.")

Archivos en DatosTarea2\UserBDSumm\v11: ['US-1.txt', 'US-2.txt', 'US-3.txt']
Archivos en DatosTarea2\UserBDSumm\v12: ['US-1.txt', 'US-2.txt', 'uS-3.txt']
Archivos en DatosTarea2\UserBDSumm\v13: ['US-1.txt', 'US-2.txt', 'US-3.txt']
Archivos en DatosTarea2\UserBDSumm\v14: ['US-1.txt', 'US-2.txt', 'US-3.txt']
Archivos en DatosTarea2\UserBDSumm\v15: ['US-1.txt', 'US-2.txt', 'US-3.txt']
Archivos en DatosTarea2\UserBDSumm\v16: ['US-1.txt', 'US-2.txt', 'US-3.txt']
Archivos en DatosTarea2\UserBDSumm\v17: ['US-1.txt', 'US-2.txt', 'US-3.txt']
Archivos en DatosTarea2\UserBDSumm\v18: ['US-1.txt', 'US-2.txt', 'US-3.txt']
Archivos en DatosTarea2\UserBDSumm\v19: ['US-1.txt', 'US-2.txt', 'US-3.txt']
Archivos en DatosTarea2\UserBDSumm\v20: ['US-1.txt', 'US-2.txt', 'US-3.txt']
Archivos en DatosTarea2\UserBDSumm\v21: ['US-1.txt', 'US-2.txt', 'US-3.txt']
Archivos en DatosTarea2\UserBDSumm\v22: ['US-1.txt', 'US-2.txt', 'US-3.txt']
Archivos en DatosTarea2\UserBDSumm\v23: ['US-1.txt', 'US-2.txt', 'US-3.txt']

In [13]:
# Se procesan las anotaciones (sumarios humanos) de OpenVideo:

# Se define la ruta de las anotaciones
user_openvideo_path = os.path.join(datasetDir, "UserOpenVideoSummary")

# Se utiliza un diccionario para almacenar las anotaciones
openvideo_annotations = {}

# Lectura de anotaciones
for folder in os.listdir(user_openvideo_path):
    folder_path = os.path.join(user_openvideo_path, folder)

    # Se mete en carpeta v21 a la v70
    if os.path.isdir(folder_path):
        user_annotations = {}

        for user_folder in os.listdir(folder_path):
            user_folder_path = os.path.join(folder_path, user_folder)

            # Se mete en carpeta user1 a user5
            if os.path.isdir(user_folder_path):
                files = os.listdir(user_folder_path)
                print(f"Archivos en {user_folder_path}: {files}")

                # Filtrar archivos de frames con .jpg y .jpeg
                frames = [
                    int(f.replace("Frame", "").replace(".jpg", "").replace(".jpeg", "").replace(".JPG", "").replace(".JPEG", "").strip())
                    for f in files
                    if f.lower().startswith("frame") and (f.lower().endswith(".jpg") or f.lower().endswith(".jpeg"))
                ]

                # Se almacena las anotaciones de frames de un usuario en especifico
                user_annotations[user_folder] = frames

        openvideo_annotations[folder] = user_annotations

print(f"\nAnotaciones de OpenVideo: {len(openvideo_annotations)} videos procesados.")

Archivos en DatosTarea2\UserOpenVideoSummary\v21\user1: ['Frame1081.jpeg', 'Frame121.jpeg', 'Frame2041.jpeg', 'Frame2101.jpeg', 'Frame2521.jpeg', 'Frame2731.jpeg', 'Frame301.jpeg', 'Frame421.jpeg', 'Frame721.jpeg', 'Frame841.jpeg', 'Frame931.jpeg']
Archivos en DatosTarea2\UserOpenVideoSummary\v21\user2: ['Frame2101.jpeg', 'Frame301.jpeg', 'Frame3060.jpeg', 'Frame780.jpeg', 'Frame961.jpeg']
Archivos en DatosTarea2\UserOpenVideoSummary\v21\user3: ['Frame1321.jpeg', 'Frame1771.jpeg', 'Frame1921.jpeg', 'Frame2101.jpeg', 'Frame2671.jpeg', 'Frame2912.jpeg', 'Frame631.jpeg', 'Frame841.jpeg', 'Frame961.jpeg']
Archivos en DatosTarea2\UserOpenVideoSummary\v21\user4: ['Frame1350.jpeg', 'Frame1591.jpeg', 'Frame1711.jpeg', 'Frame2041.jpeg', 'Frame2731.jpeg', 'Frame301.jpeg', 'Frame3211.jpeg', 'Frame721.jpeg', 'Frame871.jpeg']
Archivos en DatosTarea2\UserOpenVideoSummary\v21\user5: ['Frame2011.jpeg', 'Frame2101.jpeg', 'Frame3151.jpeg', 'Frame571.jpeg', 'Frame961.jpeg']
Archivos en DatosTarea2\UserOp

### Procesar anotaciones con K-means

In [14]:
# OBJETIVO FUNCIÓN: Aplicar K-means a una colección de descriptores y evaluar con anotaciones humanas
def calculate_kmeans(descriptors_by_video, annotations, ks):
    results = {}  # Diccionario para almacenar los valores de K y mean-f-scores

    # DEBUGGING: Imprimir las claves de las anotaciones y los descriptores
    print("Claves en las anotaciones:", list(annotations.keys()))
    print("Claves en los descriptores:", list(descriptors_by_video.keys()))

    # Se itera sobre los valores de K
    for k in ks:
        mean_fscore = 0 # Suma acumulada del Mean-F-Score para el K
        video_count = 0 # Contador de videos procesados para el K

        # Se procesa cada video en el diccionario de descriptores
        for video_id, video_descriptors in descriptors_by_video.items():
            # Ignorar videos sin anotaciones (por si acaso)
            if video_id not in annotations:
                continue

            # Se aplica K-means a los descriptores del video
            kmeans = KMeans(n_clusters=k, random_state=0).fit(video_descriptors)

            # Se calculan las distancias entre los centroides y los descriptores
            distances = cdist(kmeans.cluster_centers_, video_descriptors)

            # Se seleccionan los índices de los frames más cercanos a los centroides (keyframes)
            keyframe_indices = np.argmin(distances, axis=1)

            # DEBUGGING: Imprimir índices seleccionados y total de frames
            print(f"Indices seleccionados por Best-K-means: {keyframe_indices}")
            print(f"Total de frames en el video {video_id}: {len(video_descriptors)}")

            # Se evalua la selección de keyframes comparándola con las anotaciones humanas
            fscore_sum = 0  # Suma acumulada de f-scores para el video
            user_count = 0  # Contador de usuarios con anotaciones en el video

            for user, user_frames in annotations[video_id].items():
                # Se crean etiquetas binarias para las anotaciones
                # 1 = frame relevante (está en las anotaciones), 0 = no relevante
                y_true = np.zeros(len(video_descriptors))
                y_true[user_frames] = 1

                # Se crean etiquetas binarias para los keyframes seleccionados 
                # 1 = frame seleccionado como keyframe, 0 = no seleccionado
                y_pred = np.zeros(len(video_descriptors))
                y_pred[keyframe_indices] = 1

                # Se calcula el f-score para el usuario
                fscore = f1_score(y_true, y_pred)
                fscore_sum += fscore
                user_count += 1

            # Se promedia el f-score por usuario y se acumula
            if user_count > 0:
                mean_fscore += fscore_sum / user_count
                video_count += 1

        # Se calcula el Mean-F-Score para el K
        if video_count > 0:
            results[k] = mean_fscore / video_count
            print(f"K={k}, Mean F-score={results[k]:.4f}")

    return results

In [15]:
# Ahora se va a reasociar los descriptores con sus videos originales para mantener la relación con los videos a los que pertenecen
# Si no, al calcular K-means se podría agrupar frames de diferentes videos en los mismos clusters y esto no puede ser

# Diccionarios de descriptores por video
bdsumm_descriptors_by_video = {}
openvideo_descriptors_by_video = {}

# Reconstruir la estructura original de descriptores por video para BDSumm
start = 0
for idx, frames in enumerate(bdsumm_frames):
    end = start + len(frames)
    video_id = f"v{11 + idx}"  # Ajustar claves como v11, v12, ...
    bdsumm_descriptors_by_video[video_id] = bdsumm_descriptors[start:end]
    start = end

# Reconstruir la estructura original de descriptores por video para OpenVideo
start = 0
for idx, frames in enumerate(openvideo_frames):
    end = start + len(frames)
    video_id = f"v{21 + idx}"  # Ajustar claves como v21, v22, ...
    openvideo_descriptors_by_video[video_id] = openvideo_descriptors[start:end]
    start = end

In [16]:
# Valores de K a probar
ks = [3, 5, 7]

# Calcular resultados para BDSumm
print("\nCalculando resultados para BDSumm...")
bdsumm_results = calculate_kmeans(bdsumm_descriptors_by_video, bdsumm_annotations, ks)

# Calcular resultados para OpenVideo
print("\nCalculando resultados para OpenVideo...")
openvideo_results = calculate_kmeans(openvideo_descriptors_by_video, openvideo_annotations, ks)

print("\nResultados Finales:")
print("BDSumm:", bdsumm_results)
print("OpenVideo:", openvideo_results)



Calculando resultados para BDSumm...
Claves en las anotaciones: ['v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19', 'v20', 'v21', 'v22', 'v23', 'v24', 'v25', 'v26', 'v27', 'v28', 'v29', 'v30']
Claves en los descriptores: ['v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19', 'v20', 'v21', 'v22', 'v23', 'v24', 'v25', 'v26', 'v27', 'v28', 'v29', 'v30']
Indices seleccionados por Best-K-means: [1572  161  888]
Total de frames en el video v11: 2342
Indices seleccionados por Best-K-means: [1257  199   58]
Total de frames en el video v12: 1505
Indices seleccionados por Best-K-means: [1750 3239  979]
Total de frames en el video v13: 3918
Indices seleccionados por Best-K-means: [2024 3419 2835]
Total de frames en el video v14: 3452
Indices seleccionados por Best-K-means: [1757  773  215]
Total de frames en el video v15: 2558
Indices seleccionados por Best-K-means: [2582 1699 1104]
Total de frames en el video v16: 2650
Indices seleccionados por Best-K-means: [3610 3175 2992

### Determinar el mejor valor de K

In [17]:
# Se determina el mejor K
bdsumm_best_k = max(bdsumm_results, key=bdsumm_results.get)
openvideo_best_k = max(openvideo_results, key=openvideo_results.get)

print("\nResultados Finales:")
print(f"Mejor K para BDSumm: {bdsumm_best_k}, F-score: {bdsumm_results[bdsumm_best_k]:.4f}")
print(f"Mejor K para OpenVideo: {openvideo_best_k}, F-score: {openvideo_results[openvideo_best_k]:.4f}")


Resultados Finales:
Mejor K para BDSumm: 3, F-score: 0.0047
Mejor K para OpenVideo: 7, F-score: 0.0025


### 1.1 Comentarios de los resultados
Esto indica que en colecciones menos complejas, como BDSumm, se requiere un menor número de clusters para representar de manera efectiva los keyframes, ya que con pocos centroides es posible agrupar los descriptores más representativos (que captura lo principal de los videos). En cambio, en OpenVideo al ser una colección más diversa se beneficia de un mayor valor de K, ya que un mayor número de clusters implica los descriptores se dividan en partes más pequeñas (de forma más específica), permitiendo que los centroides representen mejor estas regiones.

A pesar de esto, los valores de Mean-F-Score obtenidos son bajos en ambas colecciones, lo que podría ser que en las anotaciones humanas hay mucha variación en comparación a los sumarios generados. Esto dificulta que los centroides se alineen con los keyframes que fueron seleccionados manualmente. Aún así se logró realizar el ejercicio con resultados que permiten entender que la complejidad de las colecciones de datos influye en la cantidad de clusters necesarios para representar de manera efectiva los keyframes.

## Ejercicio 2
Implementar dos baselines para obtener los sumarios estáticos de las colecciones BDSumm y
OpenVideo. Los métodos baselines a implementar son: Random Sampling y Uniform Sampling.

### Baseline RS

In [18]:
# OBJETIVO FUNCIÓN: Implementar Random Sampling (RS)
def random_sampling(video_descriptors, k):
    # K key-frames aleatorios
    return np.random.choice(len(video_descriptors), size=k, replace=False)

### Baseline US

In [19]:
# OBJETIVO FUNCIÓN: Implementar Uniform Sampling (US)
def uniform_sampling(video_descriptors, k):
    # K key-frames tomados de tiempos equidistantes en el video

    # Se calcula la distancia entre los índices seleccionados
    step = max(len(video_descriptors) // k, 1)
    
    # Se seleccionan os índices equidistantes
    return np.arange(0, len(video_descriptors), step)[:k]

### 2.1 Evaluación comparativa
Realizar comparación entre los baselines y best-Kmeans basado en el mean-f-score. Para una comparación justa considere usar el mismo valor de K para los baselines (el mejor K obtenido de la pregunta 1).

In [52]:
# OBJETIVO FUNCIÓN: Evaluación de Baselines y Best-K-means
def evaluate_baselines_and_kmeans(descriptors_by_video, annotations, best_k):
    # Diccionario para almacenar los resultados de la evaluación
    results = {"video_id": [], "method": [], "fscore": []}

    # Se itera sobre cada video aplicando los métodos en los descriptores
    for video_id, video_descriptors in descriptors_by_video.items():
        if video_id not in annotations:
            continue

        # Se obtienen las anotaciones (ground truth) del video actual
        ground_truth = annotations[video_id]

        # Se aplican los métodos Random Sampling y Uniform Sampling
        methods = {
            "Random Sampling": random_sampling(video_descriptors, best_k),
            "Uniform Sampling": uniform_sampling(video_descriptors, best_k),
        }

        # Se aplica Best-K-means (como en el ejercicio 1)
        kmeans = KMeans(n_clusters=best_k, random_state=0).fit(video_descriptors)
        distances = cdist(kmeans.cluster_centers_, video_descriptors)
        methods["Best-K-means"] = np.argmin(distances, axis=1)

        # Se evalua cada método comparando los keyframes seleccionados con las anotaciones humanas
        for method, keyframe_indices in methods.items():
            user_fscores = []

            # Evaluar F-score para cada usuario
            for user, user_frames in ground_truth.items():
                # Se crean etiquetas binarias para las anotaciones y keyframes seleccionados (como en el ejercicio 1)
                y_true = np.zeros(len(video_descriptors))
                y_true[user_frames] = 1
                y_pred = np.zeros(len(video_descriptors))
                y_pred[keyframe_indices] = 1

                # Se calcula el f-score para el usuario
                fscore = f1_score(y_true, y_pred)
                user_fscores.append(fscore)

            # Se calcula y se guarda el resultado promedio por usuario
            if user_fscores:
                mean_fscore = np.mean(user_fscores)
                results["video_id"].append(video_id)
                results["method"].append(method)
                results["fscore"].append(mean_fscore)

    # Se convierten los resultados a DataFrame para mostrarlos como tabla en la salida
    results_df = pd.DataFrame(results)
    return results_df


### 2.2 Tablas comparativas de los resultados

In [53]:
# Se comparan resultados en BDSumm
print("\nEvaluando en BDSumm...")
bdsumm_results_df = evaluate_baselines_and_kmeans(bdsumm_descriptors_by_video, bdsumm_annotations, bdsumm_best_k)

# Se comparan resultados en OpenVideo
print("\nEvaluando en OpenVideo...")
openvideo_results_df = evaluate_baselines_and_kmeans(openvideo_descriptors_by_video, openvideo_annotations, openvideo_best_k)


Evaluando en BDSumm...

Evaluando en OpenVideo...


In [56]:
# Mostrar tabla de resultados para BDSumm
print("\nResultados de Evaluación en BDSumm:")
print(bdsumm_results_df)


Resultados de Evaluación en BDSumm:
   video_id            method    fscore
0       v11   Random Sampling  0.000000
1       v11  Uniform Sampling  0.000000
2       v11      Best-K-means  0.000000
3       v12   Random Sampling  0.000000
4       v12  Uniform Sampling  0.000000
5       v12      Best-K-means  0.000000
6       v13   Random Sampling  0.000000
7       v13  Uniform Sampling  0.000000
8       v13      Best-K-means  0.000000
9       v14   Random Sampling  0.000000
10      v14  Uniform Sampling  0.000000
11      v14      Best-K-means  0.000000
12      v15   Random Sampling  0.000000
13      v15  Uniform Sampling  0.000000
14      v15      Best-K-means  0.000000
15      v16   Random Sampling  0.000000
16      v16  Uniform Sampling  0.000000
17      v16      Best-K-means  0.000000
18      v17   Random Sampling  0.000000
19      v17  Uniform Sampling  0.000000
20      v17      Best-K-means  0.000000
21      v18   Random Sampling  0.000000
22      v18  Uniform Sampling  0.000000
23 

In [ ]:
# Mostrar tabla de resultados para OpenVideo
print("\nResultados de Evaluación en OpenVideo:")
print(openvideo_results_df)

# Como el la tabla es tan grande para ver lo resultados se tiene que crear archivos de salida
# Se guarda la tabla en un archivo de texto
with open("openvideo_results.txt", "w") as file:
    file.write(openvideo_results_df.to_string(index=False))
print("Resultados guardados en 'openvideo_results.txt'.")

# Se guarda la tabla en un archivo CSV
openvideo_results_df.to_csv("openvideo_results.csv", index=False)
print("Resultados guardados en 'openvideo_results.csv'.")



Resultados de Evaluación en OpenVideo:
    video_id            method  fscore
0        v21   Random Sampling     0.0
1        v21  Uniform Sampling     0.0
2        v21      Best-K-means     0.0
3        v22   Random Sampling     0.0
4        v22  Uniform Sampling     0.0
..       ...               ...     ...
145      v69  Uniform Sampling     0.0
146      v69      Best-K-means     0.0
147      v70   Random Sampling     0.0
148      v70  Uniform Sampling     0.0
149      v70      Best-K-means     0.0

[150 rows x 3 columns]
Resultados guardados en 'openvideo_results.txt'.
Resultados guardados en 'openvideo_results.csv'.


### 2.3 Comentarios de resultados
Como se puede notar, la mayoría de los videos obtuvieron un F-Score de 0 para los métodos evaluados en ambas colecciones (con excepciones en algunos casos). En BDSumm el Best-K-Means logró un buen desempeño en el video v27, mientras que Random Sampling se destacó en el video v25. En OpenVideo el Best-K-Means mostró un buen desempeño en videos como v33, v36, v56, y v57, superando a los baselines en varios casos. Random Sampling obtuvo buenos valores en los videos v43 y v51, mientras que Uniform Sampling solo logró un valor no nulo en el video v65. Es dificil analizar los resultados obtenidos considerando que hay muchos valores nulos pero de todas formas se puede decir que estos resultados reflejan la ventaja de Best-K-Means en identificar keyframes relevantes, aunque los baselines pueden funcionar en circunstancias específicas.

El alto porcentaje de F-Scores de 0 se puede deber a la alta variabilidad en las anotaciones humanas como fue mencionado anteriormente en el ejercicio 1, esto dificulta la coincidencia con los keyframes seleccionados automáticamente. Los métodos baselines al no adaptarse a los descriptores de los videos, tienen problemas para identificar la información importante. Por otro lado, aunque Best-K-Means si utiliza descriptores, su desempeño depende de qué tan buenos sean estos y del valor de K elegido.